# Exercise 15 - Word2vec using Skip-gram and Negative Sampling, GloVE embedding, Embeddings projector

The goal of this exercise is to become familiar with an implementation of the word2vec algorithm using negative sampling, implemented in Keras. Make sure that you understand how the training dataset is being prepared by building a vocabulary and then encoding each word of the training corpus as an integer. You can experiment with the number of words in the vocabulary and the size of the context window (and, of course, the number of training epochs and the batch size).

- Recommended Hardware accelerator: **T4 GPU**

**Step 1:**

First, prepare the training dataset and Python dictionaries to map between words and ints

In [ ]:
!wget https://pdl-doulos.s3.us-west-2.amazonaws.com/Shakespeare.txt

In [ ]:
import random
import re
import os
import numpy as np
import keras
from keras.models    import Model
from keras.layers    import Input, Embedding, Flatten, Dense, Activation, dot

keras.backend.clear_session()

# Load the corpus as a plain text file
text = open('Shakespeare.txt', 'r').read()

print('Raw corpus length =', len(text.split()), 'words')

# Strip out the boilerplate code
t = ''
go = False
cont = False
for line in text.splitlines():
    if re.match(r'.*THE SONNETS.*', line):
        go, cont = True, True
    if re.match(r'.*<<.*', line):
        cont = False
    if re.match(r'.*>>.*', line):
        cont = True
    if go and cont:
        t += line
        t += '\n'

# Convert to lowercase
text = t.lower()

# Replace punctuation with spaces
text = text.replace('.', ' ')
text = text.replace(',', ' ')
text = text.replace(';', ' ')
text = text.replace(':', ' ')
text = text.replace('?', ' ')
text = text.replace('!', ' ')
text = text.replace('-', ' ')
text = text.replace('\t', ' ')
text = text.replace('\n', ' ')

# Strip out everything but letters and spaces
text = ''.join([text[i] for i in range(len(text)) if text[i] in ' abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'])

# Split corpus text into words
text_as_words = text.split()
print('Stripped corpus length =', len(text_as_words), 'words')

# Build list of word frequency counts
word_count = dict()
for w in text_as_words:
    if w in word_count:
        word_count[w] += 1
    else:
        word_count[w] = 1

sorted_word_count = sorted(word_count.items(), key=lambda kv: kv[1], reverse=True)

n_words = len(sorted_word_count)
print('Number of distinct words =', n_words)

# Select a vocabulary
vocab = sorted_word_count[:int(n_words * 0.2)]
vocab_size = len(vocab)
print('Vocabulary size =', vocab_size, 'words')

# -1 => out-of-vocabulary
int_to_word = [vocab[i][0] for i in range(len(vocab))]
word_to_int = dict([(int_to_word[i], i) for i in range(len(vocab))])

# Convert corpus from words to ints
text_as_ints = [word_to_int[w] if w in word_to_int else -1 for w in text_as_words]
#print(text_as_ints[10000:10500])

**Step 2:**

Build and save a Keras model

In [ ]:
def gen_dataset(n_samples):
    # Build a dataset consisting of pairs of words, a target word and a context/non-context word

    window = [-5,-4,-3,-2,-1,+1,+2,+3,+4,+5]

    window_len = len(window) // 2
    target_data = []
    context_data = []
    label = []
    count = 0
    while count < n_samples:
        # Pick a target word
        target_word = -1
        while target_word == -1:
            # Make sure we don't overflow when considering a window around the target word
            target_ix = random.randrange(window_len,len(text_as_ints)-window_len)
            target_word = text_as_ints[target_ix]
        if random.randrange(2) == 1:
            # Pick a context word with probability = 1/frequency
            candidates = []
            weights = []
            for offset in window:
                intval = text_as_ints[target_ix + offset]
                if intval != -1:
                    candidates.append(intval)
                    weights.append(1/vocab[intval][1])
            if len(candidates) == 0: continue
            context_word = random.choices(candidates, weights=weights)[0]
            #print('Choice', candidates, context_word)
            label.append(1)
        else:
            # Pick a non-context word from the vocabulary
            done = False
            while not done:
                intval = random.randrange(0,len(int_to_word))  # -1 => out-of-vocabulary
                done = True
                for offset in window:
                    if intval == text_as_ints[target_ix + offset]: done = False
                context_word = intval
            label.append(0)
        target_data.append(target_word)
        context_data.append(context_word)
        count += 1

    x_target = np.array(target_data)
    x_context = np.array(context_data)
    y = np.array(label)  # Convert labels to a NumPy array
    return (x_target, x_context, y) # Return a tuple of NumPy arrays


# Build Keras model for Skip-gram Word2vec with negative sampling

target_in  = Input(shape=[1], name='target_in')
context_in = Input(shape=[1], name='context_in')

embedding   = Embedding(vocab_size, 256, name='word_embed')
target_emb  = embedding(target_in)
context_emb = embedding(context_in)

x = dot([target_emb, context_emb], axes=2, normalize=True)
x = Flatten()(x)
x = Dense(1, activation='sigmoid')(x)

model = Model(inputs=[target_in, context_in], outputs=x)
model.summary()

model.compile(loss='binary_crossentropy', optimizer='Adam', metrics=['accuracy'])

model.save('word2vec_after_0_epochs.keras')

In [ ]:
keras.backend.clear_session()

# Load the model without compiling it
model = keras.models.load_model('word2vec_after_0_epochs.keras', compile=False)

# Re-compile the model with the Adam optimizer
model.compile(loss='binary_crossentropy', optimizer='Adam', metrics=['accuracy'])


for i in range(10):
    x_target_data, x_context_data, y_train = gen_dataset(1000000)
    # Pass inputs as a list
    model.fit(x=[x_target_data, x_context_data], y=y_train, epochs=1, batch_size=128)

model.save('word2vec_after_10_epochs.keras')

**Step 3:**

- Access the content and shape of the trained embedding layer
- Determine cosine similarity for two sets of words from the trained embedding

In [ ]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

embedding_layer = model.get_layer('word_embed')

# Get the embedding matrix
embedding_array = embedding_layer.get_weights()[0]

# `embeddings` has a shape of (vocabulary_size, embedding_dim)
print("Shape of the embedding matrix:", embedding_array.shape)

# `word_to_int` is a mapping (i.e. dict) from words to their index
words_embeddings = {w:embedding_array[idx] for w, idx in word_to_int.items()}

vector1 = words_embeddings['winter']
vector2 = words_embeddings['summer']

cosine_similarity_1 = dot(vector1, vector2) / (norm(vector1) * norm(vector2))

print ('cosine_similarity for winter and summer is ', cosine_similarity_1)

vector3 = words_embeddings['love']
vector4 = words_embeddings['winter']

cosine_similarity_2 = dot(vector3, vector4) / (norm(vector3) * norm(vector4))

print ('cosine_similarity for love and winter is ', cosine_similarity_2)

**Step 4:**

- Obtain a pretrained 50 dimension GloVe embedding (glove_50d.txt)
- Write a function to return the number of words in the embedding
- Print word vector of a word of your choice from the embedding



In [ ]:
%%bash

wget -q https://nlp.stanford.edu/data/wordvecs/glove.2024.wikigiga.50d.zip
unzip -q glove.2024.wikigiga.50d.zip
cp wiki_giga_2024_50_MFT20_vectors_seed_123_alpha_0.75_eta_0.075_combined.txt glove_50d.txt

**Step 4(a):**

View the first 5 words and their corresponding vectors from glove_50d.txt file

In [ ]:
!head -n 5 glove_50d.txt

**Step 4(b):**

- Parse the Glove_50d text file and store the contents as float values in a Numpy array
- Use regular expression to check if there is a corresponding float vector associated with the word, otherwise skip the word

In [ ]:
import numpy as np
import re

def load_glove_model(file_path):
    print("Loading Glove Model")
    glove_model = {}
    with open(file_path,'r') as f:
        for line in f:
            split_line = line.split()
            if split_line and re.match(r'[a-zA-Z]', split_line[0]): # Check if the first element is a word
                word = split_line[0]
                try:
                    # Attempt to convert the string representations of numbers to floats
                    embedding = np.array(split_line[1:], dtype=np.float64)
                    glove_model[word] = embedding
                except ValueError:
                    # Skip lines where conversion to float fails
                    print(f"Skipping line for word '{word}' due to invalid embedding values.")
                    continue

    print(f"{len(glove_model)} words loaded!")
    return glove_model

gloveModel = load_glove_model('glove_50d.txt')


**Step 4(c):**

- Perform cosine similarity on two words using the GloVe embedding

In [ ]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

vector1 = gloveModel['car']
vector2 = gloveModel['truck']

cosine_similarity = dot(vector1, vector2) / (norm(vector1) * norm(vector2))

print ('cosine_similarity for car and truck is ', cosine_similarity)

vector3 = gloveModel['car']
vector4 = gloveModel['tulip']

cosine_similarity_1 = dot(vector3, vector4) / (norm(vector3) * norm(vector4))

print ('cosine_similarity for car and tulip is ', cosine_similarity_1)

**Further experimentation:**

- Try out [Embedding Projector](https://projector.tensorflow.org/) by using MNIST with images, T-SNE and 2D projection

- Evaluate the GloVE embedding using scripts from [GloVE GitHub](https://github.com/stanfordnlp/GloVe/tree/master/eval/python)